# Week 4 Lab: Three Ways to Wire a Machine
**The Evolution of Machine Learning and AI · SETU**

> **How this lab works.** The lecture had six parts. This lab has **six exercises, one per part, in the same order**, and each exercise does on your machine exactly what the matching slides did on the screen. About **20 minutes** each. Every exercise has the same five parts:
> 1. **From the lecture**: the slides it follows and the red box it checks
> 2. **What you are doing**, in plain English
> 3. **Run this**: one to four cells; you do not write code
> 4. **What you should see**: the printout, labelled (Printout A, Table B ...)
> 5. **Check these against `SUBMISSION.md`**: each question says exactly where to look
>
> `SUBMISSION.md` is **already filled in** with the reference answers. At the end of each exercise, check the numbers on your screen against it and change any that differ.
>
> **You do not need the booklet for the lab.** Nothing here needs the slides or the booklet either: every answer is a printout in this notebook.
> **If an exercise takes more than 40 minutes, stop and post in the channel.** That is a bug in the material, not in you.
>
> This week's cells train small networks, so a few take 20 to 40 seconds. The longest (Exercise 4's memory task) is about a minute. No PyTorch is needed.

| Exercise | Lecture part | Slides | Red box | What you run |
|---|---|---|---|---|
| 1 · A guess about the world | Part 1 | 7 to 13 | box 1 | `guess_the_rule`, `the_race`, `shuffle_test` |
| 2 · Convolution | Part 2 | 14 to 20 | box 2 | `by_hand`, `kernel_walk`, `equivariance` |
| 3 · What the guess buys | Part 3 | 21 to 25 | box 3 | `depth_reaches`, `the_head`, `no_payoff` |
| 4 · Recurrence | Part 4 | 26 to 35 | box 4 | `a_power`, `gradient_through_time`, `half_life`, `memory_task` |
| 5 · Attention | Part 5 | 36 to 45 | box 5 | `soft_lookup`, `four_instructions`, `cannot_see_order`, `the_bill` |
| 6 · The bargain | Part 6 | 46 to 50 | box 6 | `the_bargain`, `seven_questions` |

**Run the cell below first.** It loads the lab code.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

import lecture_steps as ls      # one helper per lecture idea; you never edit it
print("Lab code loaded. You are ready for Exercise 1.")

Lab code loaded. You are ready for Exercise 1.


---
# Exercise 1 · A guess about the world  *(≈ 20 min)*

### 1. From the lecture
*Part 1, slides 7 to 13.* **Red box 1:** *Any set of examples fits infinitely many rules; choosing one needs a preference that is not in the data, the inductive bias. How a model is wired is a guess about the world. A more flexible model is not a better model. The shuffle test: destroy the thing the guess is about and see whether the advantage vanishes.*

### 2. What you are doing
**In one sentence:** you will watch a small machine with a built-in guess beat a much bigger machine with no guess, and then prove that the guess was the whole reason.

Slide 7's puzzle (1, 2, 3 map to 2, 4, 6: what does 4 map to?), then slide 8's race between a 700-dial model that assumes "nearby pixels belong together" and a 16,708-dial model that assumes nothing, then slide 12's shuffle test.

### 3. Run this: the puzzle

In [2]:
ls.guess_the_rule()

Printout A: five different rules. Every one of them fits the three examples you were given.
  rule                                                 1  2  3   ->  4
  double it                                            2  4  6   ->  8
  double it, unless it is 4, then 1000                 2  4  6   ->  1000
  n + n                                                2  4  6   ->  8
  the n-th even number                                 2  4  6   ->  8
  n squared minus n squared plus 2n, unless n is 4, then 9 2  4  6   ->  9

  The examples cannot choose between them. YOU chose 'double it', because you prefer simple rules.
  That preference is the inductive bias. Without one, there is no answer to 'what comes next'.


### 4. What you should see
**Printout A**: five different rules, and every one fits the three examples. Only your *preference* for simple rules picks "double it". That preference is the inductive bias.

### 3. Run this: the race at 128 pictures  *(about 10 seconds)*

In [3]:
ls.the_race(n_train=128)

Table B: the same 128 training images, two models. Chance is 0.250. The MLP has 24 times the parameters and scores at chance on new images.


,model,parameters,train,test
0,"CNN (assumes: nearby pixels matter, same patte...",700,0.992,0.941
1,MLP (assumes nothing about space),16708,1.000,0.264


**Table B**: the tiny model scores about **0.94** on new pictures; the big one, with 24 times the dials, scores about **0.26**, which is chance (0.25). Both score 1.0 or nearly on the pictures they studied. The big one is not undertrained; it has no reason to prefer the right answer.

### 3. Run this: the shuffle test  *(about 30 seconds)*

In [4]:
ls.shuffle_test()

Table C: one fixed shuffle of the 256 pixels, applied to every image. Nothing is deleted; only 'which pixels are neighbours' is destroyed.


,pixel order,CNN test,MLP test,gap
0,original pixel order,0.992,0.328,0.664
1,pixels shuffled,0.255,0.333,-0.078


**Table C**: with the pixels shuffled (same shuffle for every picture, nothing deleted), the tiny model falls from **0.992 to 0.255**, chance. The big model does not move (0.328 to 0.333). The gap flips from **+0.664 to −0.078**.

### 5. Check these against `SUBMISSION.md`, Exercise 1
- **1a.** How many of the five rules fit the examples 1→2, 2→4, 3→6? *Where to look: Printout A.*
- **1b.** At 128 pictures, the test score of the tiny model and of the big one. *Where to look: Table B, column `test`.*
- **1c.** Was the big model undertrained? *Where to look: Table B, column `train`.*
- **1d.** After the shuffle, what does the tiny model score, and what happened to the gap? *Where to look: Table C, row "pixels shuffled".*
- **1e.** In one sentence: what was the tiny model's advantage made of? *Where to look: Table C: the advantage vanished when only "which pixels are neighbours" was destroyed.*

---
# Exercise 2 · Convolution: things near each other  *(≈ 20 min; the code on slide 19)*

### 1. From the lecture
*Part 2, slides 14 to 20.* **Red box 2:** *A convolution slides a small square of numbers (a kernel) over the picture: multiply the pairs, add. The kernel does not change as it moves (weight sharing). It assumes locality, stationarity and compositionality. Equivariance: shift the picture and the output shifts the same way, exactly.*

### 2. What you are doing
**In one sentence:** you will do a convolution by hand, watch the computer do the same thing across a whole picture, and check that moving the picture moves the answer exactly.

Slide 16's pencil calculation (the 3 × 3 patch times the edge kernel gives 3.2), slide 15's kernel walk over a small picture, and slide 18's equivariance test.

### 3. Run this: by hand

In [5]:
ls.by_hand()

Printout A: the patch (dark on the left, bright on the right) and the vertical-edge kernel
  patch:            kernel:
   0.1   0.2   0.9        -1     0     1
   0.1   0.2   0.9        -2     0     2
   0.1   0.2   0.9        -1     0     1

Printout B: multiply each pair, row by row, then add everything up
  (0.1 x -1) + (0.2 x 0) + (0.9 x 1)  =  0.8
  (0.1 x -2) + (0.2 x 0) + (0.9 x 2)  =  1.6
  (0.1 x -1) + (0.2 x 0) + (0.9 x 1)  =  0.8
  total = 3.2     <- a strong response: there IS a left-to-right edge here

Printout C: the same kernel on a flat patch (every value 0.5): total = 0.0  (the weights sum to zero, so flat gives nothing)
Printout D: the patch reversed (bright on the left): total = -3.2  (the sign says which way round the edge is)


### 4. What you should see
**Printout B**: the three rows multiply and add to 0.8, 1.6, 0.8, total **3.2**. **Printout C**: a flat patch gives **0.0** (the kernel's numbers sum to zero). **Printout D**: the patch reversed gives **−3.2** (the sign says which way round the edge is).

### 3. Run this: the kernel walk

In [6]:
ls.kernel_walk()

Printout A: a 6 x 6 image, dark on the left, bright on the right:
0 0 0 1 1 1
0 0 0 1 1 1
0 0 0 1 1 1
0 0 0 1 1 1
0 0 0 1 1 1
0 0 0 1 1 1

Table B: the feature map. The SAME nine numbers were used at every position; each output is nine multiplies and eight adds.
        Big numbers mark WHERE the edge is. Zeros everywhere the patch is flat.


,0,1,2,3
0,0,4,4,0
1,0,4,4,0
2,0,4,4,0
3,0,4,4,0


**Table B**: a 6 × 6 picture (dark left, bright right) becomes a 4 × 4 feature map with **4** in the two middle columns and **0** elsewhere. The same nine numbers were used at every position; big values mark *where* the edge is.

### 3. Run this: move the picture

In [7]:
ls.equivariance()

Table C: shift the image, then convolve; convolve, then shift the result. For a convolution the two are identical. A dense layer treats the shifted image as something new.


,operation,quantity,value,verdict
0,convolution,max |conv(shift(x)) - shift(conv(x))|,0.00e+00,"equivariant, by construction"
1,dense layer,max |W shift(x) - W x|,1.987,unrelated output (60% of scale)


**Table C**: for the convolution the difference between "shift then convolve" and "convolve then shift" is **0.00e+00**, exactly. For a dense layer it is **1.987**, about 60% of the output's size: a shifted picture is a different picture to it.

### 5. Check these against `SUBMISSION.md`, Exercise 2
- **2a.** The total for the dark-to-bright patch, for the flat patch, and for the reversed patch. *Where to look: Printouts B, C, D.*
- **2b.** Why does the flat patch give zero? *Where to look: Printout A: the kernel's nine numbers sum to zero.*
- **2c.** In the feature map, where are the non-zero numbers, and what do they mark? *Where to look: Table B.*
- **2d.** The equivariance error for the convolution and for the dense layer. *Where to look: Table C, column `value`.*
- **2e.** In one sentence: why is the convolution's error exactly zero? *Where to look: red box 2: the same nine numbers are used everywhere, so the operation cannot tell where it is.*

---
# Exercise 3 · What the guess buys, and when it does not pay  *(≈ 20 min)*

### 1. From the lecture
*Part 3, slides 21 to 25.* **Red box 3:** *Stacking layers turns local looks into a view of the whole picture. Equivariance is not invariance; the ending decides which you get (0.998 vs 0.230). The guess fails when the data has no arrangement, when position matters, when things are far apart, or when the change is not a shift. On pre-centred data it buys nothing.*

### 2. What you are doing
**In one sentence:** you will see how stacking stencils lets one number see the whole picture, how the ending of a network can keep or throw away the guess, and one case where the guess buys nothing.

Slide 22's receptive-field table, slide 23's two endings (about 35 seconds to train), and slide 24's honest counter-example (about 10 seconds).

### 3. Run this: depth reaches everything

In [8]:
ls.depth_reaches()

Table A: each layer sees only a 3 x 3 window, but stacked layers see further and further. By the last layer one unit depends on the whole image.


,layer,operation,receptive field,pixels seen,stride so far
0,1,conv 3x3,3 by 3,9,1
1,2,conv 3x3,5 by 5,25,1
2,3,pool 2x2,6 by 6,36,2
3,4,conv 3x3,10 by 10,100,2
4,5,conv 3x3,14 by 14,196,2
5,6,pool 2x2,16 by 16,256,4
6,7,conv 3x3,24 by 24,576,4
7,8,conv 3x3,32 by 32,1024,4


### 4. What you should see
**Table A**: layer 1 sees 3 × 3 pixels; by layer 8 one unit sees **32 × 32**, all 1,024 pixels. Nobody wrote a rule about the whole picture.

### 3. Run this: the two endings  *(about 35 seconds)*

In [9]:
ls.the_head()

Table B: trained with the pattern only on the LEFT of the image, tested with it only on the RIGHT. Same convolutional layers; the only difference is how the network ends.


,model,parameters,positions seen in training,positions never seen,drop
0,"CNN, global pooling",700,1.000,0.998,0.002
1,"CNN, flatten head",2716,0.993,0.230,0.763
2,MLP,16708,0.470,0.253,0.217


**Table B**: identical convolution layers. Ending that averages the map: **0.998** on positions never seen. Ending that flattens and wires every position: **0.230**, chance. The ending threw the guess away.

### 3. Run this: where it does not pay  *(about 10 seconds)*

In [10]:
ls.no_payoff()

Table C: 8 x 8 digits that somebody already centred and cropped. Translation was solved before the model saw the data, so the assumption has nothing left to buy.


,model,parameters,train acc,test acc,verdict
0,"CNN, global pooling",754,0.995,0.940,the assumption buys little here
1,MLP,4810,1.000,0.978,because the data was pre-centred for it


**Table C**: on 8 × 8 digits that were already centred and cropped, the convolution model scores **0.940** and the plain model **0.978**. Translation was solved before the model saw the data.

### 5. Check these against `SUBMISSION.md`, Exercise 3
- **3a.** How many pixels does one unit see at layer 2, and at layer 8? *Where to look: Table A, column `pixels seen`.*
- **3b.** On positions never seen: the score with the averaging ending, and with the flatten ending. *Where to look: Table B, column `positions never seen`.*
- **3c.** In one sentence: why did the flatten ending fail on the right-hand side? *Where to look: Table B; slide 23: it wired a different dial to every position.*
- **3d.** On the pre-centred digits, which model won, and why does that not contradict the week? *Where to look: Table C, column `verdict`.*

---
# Exercise 4 · Recurrence: one thing after another  *(≈ 20 min; the code on slide 34)*

### 1. From the lecture
*Part 4, slides 26 to 35.* **Red box 4:** *A recurrent network is one rulebook applied at every word, updating a fixed-size notepad. Long chains forget because the signal is multiplied by the same number at every step. An LSTM's keep-gate, chosen fresh each step, lets the signal survive. Two limits remain: fixed notepad, one step at a time.*

### 2. What you are doing
**In one sentence:** you will see why a reader with one rulebook forgets the start of a long sentence, and how a gate the reader controls fixes it.

Slide 29's arithmetic (0.9 and 1.1 to the sixtieth power), slide 30's measured signal through a real chain, slide 31's half-life table, and slide 32's memory task (about a minute).

### 3. Run this: a number to the power sixty

In [11]:
ls.a_power()

Printout A: the same number multiplied by itself 60 times
  0.5 ^ 60 =  8.67362e-19
  0.9 ^ 60 =   0.00179701
  1.0 ^ 60 =            1
  1.1 ^ 60 =      304.482
  Two numbers 0.2 apart give results 170,000 times apart. A message passed through 60 identical steps either dies or explodes.


### 4. What you should see
**Printout A**: 0.9 to the sixtieth is **0.0018**; 1.1 to the sixtieth is **304**. Two numbers 0.2 apart, results 170,000 times apart.

### 3. Run this: the signal, measured

In [12]:
ls.gradient_through_time()

Table B: a real recurrent layer, a real gradient, followed back through 60 steps. At scale 0.5 what reaches step 1 is 7e-17: in 32-bit arithmetic, zero.


,weight scale,signal left after 10 steps,signal left after 60 steps
0,0.5,3.24e-03,6.96e-17
1,0.9,7.32e-01,1.49e-01
2,1.0,7.88e-01,1.74e-01
3,1.2,2.06e+00,4.74e+00


**Table B**: at weight scale 0.5 the signal from word 1 that reaches the end after 60 steps is **6.96e-17**, zero to a computer. At 0.9 it survives (0.15). At 1.2 it grows.

### 3. Run this: the keep-gate

In [13]:
ls.half_life()

Table C: an LSTM multiplies its memory by f at every step. f is something it CHOOSES, fresh, each step. Learn f near 1 and the past survives.


,forget gate f,half-life (steps),1% left after (steps)
0,0.800,3.1,21
1,0.950,13.5,90
2,0.990,69.0,458
3,0.999,692.8,4603


**Table C**: a keep-gate of 0.80 halves the memory every 3 steps; 0.95 every 13.5; 0.999 every 693.

### 3. Run this: the memory task  *(about a minute)*

In [14]:
ls.memory_task()

Table D: remember the first symbol across T steps, chance 0.250. Read the row for length 90: the RNN's gradient has collapsed and it is near chance; the LSTM's survives and it is perfect.


,sequence length,RNN test acc,RNN grad at step 1,LSTM test acc,LSTM grad at step 1
0,10,1.000,1.90e-06,1.000,1.92e-08
1,40,1.000,2.65e-05,0.997,5.21e-09
2,90,0.463,7.81e-18,1.000,8.93e-06


**Table D**: remember the first symbol across T steps (chance 0.250). At 90 steps the plain chain scores **0.463** with a signal of **7.8e-18**; the gated one scores **1.000** with a signal of **8.9e-06**.

### 5. Check these against `SUBMISSION.md`, Exercise 4
- **4a.** 0.9 to the sixtieth power, and 1.1 to the sixtieth. *Where to look: Printout A.*
- **4b.** The signal after 60 steps at weight scale 0.5, and at 0.9. *Where to look: Table B.*
- **4c.** With a keep-gate of 0.95, after how many steps is half the memory left? *Where to look: Table C.*
- **4d.** At 90 steps: the plain chain's score and signal, and the gated one's. *Where to look: Table D, row 90.*
- **4e.** In one sentence: why does the plain chain's signal die but the gated one's does not? *Where to look: red box 4: the same number at every step versus a chosen gate near 1.*

---
# Exercise 5 · Attention: who should talk to whom  *(≈ 20 min; the code on slide 44)*

### 1. From the lecture
*Part 5, slides 36 to 45.* **Red box 5:** *Attention lets every word look at every other; content decides who listens to whom. Four steps: write three cards (query, key, value), score, share out (softmax), mix. Bare attention cannot see order, so position labels are added. Cost grows with the square of the length.*

### 2. What you are doing
**In one sentence:** you will print the table where each word decides who to listen to, walk the four steps for one word, and show that attention cannot tell what order the words came in.

Slide 38's table where every row sums to 1, slide 39's four instructions for the word "are", slide 40's shuffle test, and slide 43's bill.

### 3. Run this: every row sums to 1

In [15]:
ls.soft_lookup()

Table A: attention weights. Each ROW is one token asking 'who should I listen to?'. Every row sums to 1.
         Read the row for 'are': most of its weight is on 'keys', four positions away, not on 'left' next door.


,the,keys,that,I,left,are,sum
the,0.167,0.167,0.167,0.167,0.167,0.167,1.002
keys,0.052,0.383,0.052,0.077,0.052,0.383,0.999
that,0.167,0.167,0.167,0.167,0.167,0.167,1.002
I,0.142,0.211,0.142,0.153,0.142,0.211,1.001
left,0.069,0.418,0.057,0.343,0.057,0.057,1.001
are,0.016,0.735,0.013,0.122,0.013,0.100,0.999


### 4. What you should see
**Table A**: six words by six words, each row summing to 1 (to rounding). Row **are**: **0.735** on **keys**, four words away; **0.013** on **left**, next door.

### 3. Run this: the four instructions

In [16]:
ls.four_instructions()

Printout B: the four instructions, for the token 'are'
  1 PROJECT   'are' becomes a query (what it is looking for: a plural subject); every token offers a key (what it is).
  2 SCORE     query . key for each token:      the=0.20  keys=4.00  that=0.00  I=2.20  left=0.00  are=2.00
  3 NORMALISE softmax turns scores into shares: the=0.02  keys=0.73  that=0.01  I=0.12  left=0.01  are=0.10   (sum 1.00)
  4 MIX       output for 'are' = those shares x the value vectors: 0.86, 0.85, 0.11, 0.04
  Distance never appeared in any of the four steps. 'keys' is four positions away and wins anyway.


**Printout B**: for "are": scores (keys = 4.00, the highest), shares after softmax (keys = 0.73, sum 1.00), and the mixed output. Distance never appeared.

### 3. Run this: attention cannot see order

In [17]:
ls.cannot_see_order()

Table C: without position information, shuffling the tokens changes the output by 8.88e-16, which is floating-point rounding. With it, the output changes.


,input,max |shuffle(attend(x)) - attend(shuffle(x))|,verdict
0,embeddings only,8.88e-16,order is invisible: a bag of tokens
1,embeddings + positional encoding,4.192,order now changes the answer


**Table C**: shuffle the words, attend, un-shuffle: the difference is **8.88e-16** (rounding error) without position labels, **4.192** with them.

### 3. Run this: the bill

In [18]:
ls.the_bill()

Table D: attention reaches any distance in ONE step, but its cost grows with the SQUARE of the length. At 32,768 tokens it is 65 times an RNN.


,sequence length,RNN ops,CNN ops,attention ops,attention / RNN,longest path (attention),longest path (RNN)
0,64,1.68e+07,5.03e+07,1.89e+07,1.1x,1,64
1,512,1.34e+08,4.03e+08,2.68e+08,2.0x,1,512
2,4096,1.07e+09,3.22e+09,9.66e+09,9.0x,1,4096
3,32768,8.59e+09,2.58e+10,5.58e+11,65.0x,1,32768


**Table D**: at 64 words attention costs 1.1 times a recurrent chain; at 32,768 words, **65 times**. Its longest path is always **1**; the chain's is the length.

### 5. Check these against `SUBMISSION.md`, Exercise 5
- **5a.** In row "are", the share on "keys" and on "left". *Where to look: Table A, row `are`.*
- **5b.** What does each row sum to? *Where to look: Table A, column `sum`.*
- **5c.** The four instructions, in order. *Where to look: Printout B.*
- **5d.** The shuffle difference without and with position labels. *Where to look: Table C, column `max ...`.*
- **5e.** At 32,768 words, how much more does attention cost than the chain, and what is its longest path? *Where to look: Table D, last row.*

---
# Exercise 6 · The bargain  *(≈ 20 min)*

### 1. From the lecture
*Part 6, slides 46 to 50.* **Red box 6:** *A true guess built into the wiring substitutes for data. Little data: the well-guessed machine wins. Enormous data: the machine that guesses less can learn the structure and then wins. The Vision Transformer shows it; the bitter lesson, read properly, is about where knowledge enters.*

### 2. What you are doing
**In one sentence:** you will measure the bargain from Exercise 1 at four sizes of data and see that the guess is worth more the less data you have.

Slide 47's bargain, measured: the same two models as Exercise 1 at four data sizes (about 30 seconds), then slide 50's seven questions.

### 3. Run this: the bargain, measured  *(about 30 seconds)*

In [19]:
ls.the_bargain()

Table A: the same two models as Exercise 1, at four data sizes. The structured model wins by 47 to 75 points at every size here. With 128 times the data (Exercise 1's slide), the unstructured one catches up.


,train images,CNN train,CNN test,MLP train,MLP test,CNN advantage
0,64,0.859,0.723,1.0,0.252,+47.2 pts
1,128,0.992,0.945,1.0,0.275,+67.0 pts
2,256,1.000,0.997,1.0,0.252,+74.5 pts
3,512,1.000,0.992,1.0,0.328,+66.3 pts


### 4. What you should see
**Table A**: at 64, 128, 256, 512 pictures the tiny model leads by **47 to 75 points** every time, and the big one sits at chance with 1.000 on training. The slide's second half (128 times the data) is the part a laptop cannot afford; the Vision Transformer at 300 million pictures is the same curve, further along.

### 3. Run this: the seven questions

In [20]:
ls.seven_questions()

Printout B: seven questions for anything you design
  1. What is connected to what, in one layer?
  2. Which weights are shared, and across what: space, time, or nothing?
  3. What does that combination assume, as a sentence about the world?
  4. Is that sentence true of my data, and how would I know? (the shuffle test costs one line)
  5. Does my architecture carry the assumption all the way to the output? (0.998 against 0.230)
  6. What is the cost, and where is the crossover?
  7. How much data do I have, relative to the structure to be learned?


**Printout B**: the seven questions for anything you design.

### 5. Check these against `SUBMISSION.md`, Exercise 6
- **6a.** The tiny model's advantage at 64 pictures and at 512. *Where to look: Table A, column `CNN advantage`.*
- **6b.** What was the big model's training accuracy at every size? *Where to look: Table A, column `MLP train`.*
- **6c.** In one sentence: at what point does the bargain flip, and what flips it? *Where to look: red box 6 and slide 48: enough data for the unstructured model to learn the structure.*
- **6d.** Which of the seven questions does Exercise 1's shuffle test answer? *Where to look: Printout B.*
- **6e.** Which does Exercise 3's two endings answer? *Where to look: Printout B.*

**That is the whole lab.** Put your name on `SUBMISSION.md`, check it matches your run, and upload it.

---
# Optional: going further  *(not required · only if you want more)*

- **The browser toys.** `interactive/convolution-feature-map-visualiser.html`, `attention-visualiser.html`, `sequence-memory-explorer.html`. Double-click; no install.
- **Break it on purpose.** (1) In `inductive_bias.py`, shuffle whole columns only and predict what the CNN does before running. (2) In `sequences.py`, set the LSTM's `forget_bias` to 0.0 and explain the failure with the half-life table. (3) In `attention.py`, remove the `sqrt(d)` and raise `d_model` to 128; name the failure.
- **The full run.** `python run_lab.py` reproduces every number in the lecture (about four minutes; `--quick` for one).
- **PyTorch.** `torch_models.py` builds the same three families in PyTorch, with parameter counts side by side. Optional.
- **Read more.** The full booklet has every derivation. The core booklet is the required reading; neither is needed for this lab.